# Demo MedGemma en direct via Streamlit (Colab)

> **Prototype pedagogique. Non destine au diagnostic.**

Ce notebook clone TON depot, charge MedGemma, ecrit l'app Streamlit et ouvre un tunnel public pour la demo.

**Avant de lancer :** `Execution > Modifier le type d'execution > GPU` (T4).
Puis lance les cellules **dans l'ordre**.

## 1. Cloner ton depot + installer + outil de tunnel

In [ ]:
import os
if not os.path.exists('DS-7C-assistant-radiologue-virtuel'):
    !git clone https://github.com/TonneauBenjamin/DS-7C-assistant-radiologue-virtuel.git
%cd /content/DS-7C-assistant-radiologue-virtuel
!pip install -q -U transformers accelerate bitsandbytes sentencepiece streamlit kagglehub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!nvidia-smi -L

## 2. Connexion Hugging Face
MedGemma est *gated* : accepte les conditions sur https://huggingface.co/google/medgemma-4b-it, puis colle un token *read*.

In [ ]:
from huggingface_hub import login
login()  # colle ton token hf_...

## 3. (Optionnel) Recuperer 6 vraies radios pour la demo
Pour pouvoir tester sans avoir d'images sur ton PC, on met quelques radios dans `data/real_cases/`. Ignore cette cellule si tu preferes televerser tes propres images.

In [ ]:
try:
    import kagglehub, shutil, random
    from pathlib import Path
    kagglehub.login()  # username + key (kaggle.json)
    base = Path(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia'))
    test_root = next(base.rglob('test'))
    random.seed(0)
    dst = Path('data/real_cases'); dst.mkdir(parents=True, exist_ok=True)
    picked = []
    for sub, label in [('NORMAL', 'normal'), ('PNEUMONIA', 'suspected_opacity')]:
        fs = sorted((test_root / sub).glob('*.jpeg')); random.shuffle(fs)
        for i, f in enumerate(fs[:3]):
            out = dst / f'CXR_REAL_{label}_{i}{f.suffix.lower()}'; shutil.copy(f, out); picked.append(out.name)
    print('Images de demo pretes :', picked)
except Exception as e:
    print('Etape optionnelle ignoree :', e)

## 4. Ecrire l'app Streamlit dans le depot
L'app charge MedGemma (8-bit), reutilise tes garde-fous, et permet de choisir une radio d'exemple ou d'en televerser une.

In [ ]:
%%writefile app_medgemma_colab.py
"""Demo Streamlit branchee sur MedGemma - a lancer DANS Colab (GPU requis).

Prototype pedagogique. Non destine au diagnostic.
"""
from __future__ import annotations

import time
from pathlib import Path

import streamlit as st
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

from src.guardrails import apply_safety_guardrails, WARNING_TEXT

MODEL_ID = "google/medgemma-4b-it"

st.set_page_config(page_title="Assistant radiologue - MedGemma", layout="wide")
st.title("Assistant radiologue virtuel - MedGemma")
st.warning(
    "Prototype pedagogique. Non destine au diagnostic. "
    "Validation par un professionnel qualifie requise."
)


@st.cache_resource(show_spinner="Chargement de MedGemma en 8-bit (une seule fois, ~quelques minutes)...")
def load_model():
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=BitsAndBytesConfig(load_in_8bit=True),
        device_map="auto",
    )
    return processor, model


PROMPTS = {
    "baseline": {
        "name": "baseline",
        "system": "You are an expert radiologist.",
        "user": "Look at this chest X-ray. Answer with exactly one word: NORMAL if the "
                "lungs are clear, or PNEUMONIA if there is a lung opacity or consolidation.",
        "max_new_tokens": 10,
    },
    "improved": {
        "name": "improved",
        "system": "You are an expert chest radiologist. Missing a pneumonia is dangerous, "
                  "so when there is ANY sign of opacity, consolidation or infiltrate, classify as PNEUMONIA.",
        "user": "Analyze this chest X-ray step by step, then end with a final line exactly in "
                "the form \"FINAL: NORMAL\" or \"FINAL: PNEUMONIA\".",
        "max_new_tokens": 160,
    },
}


def predire(image, cfg, processor, model):
    eot = processor.tokenizer.convert_tokens_to_ids("<end_of_turn>")
    pad = processor.tokenizer.pad_token_id or 0
    messages = [
        {"role": "system", "content": [{"type": "text", "text": cfg["system"]}]},
        {"role": "user", "content": [
            {"type": "text", "text": cfg["user"]},
            {"type": "image", "image": image},
        ]},
    ]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    t0 = time.time()
    with torch.inference_mode():
        gen = model.generate(**inputs, max_new_tokens=cfg["max_new_tokens"], do_sample=False,
                             eos_token_id=eot, pad_token_id=pad,
                             repetition_penalty=1.3, no_repeat_ngram_size=3)
    latency_ms = int((time.time() - t0) * 1000)
    txt = processor.decode(gen[0][input_len:], skip_special_tokens=True)
    up = txt.upper()
    if "FINAL:" in up:
        up = up.split("FINAL:")[-1]
    if "PNEUMON" in up:
        pred, conf = "suspected_opacity", 0.80
    elif "NORMAL" in up:
        pred, conf = "normal", 0.75
    else:
        pred, conf = "uncertain", 0.40
    result = {
        "image_quality": "good", "predicted_class": pred, "confidence": conf,
        "visual_evidence": [txt.strip()[:160]] if txt.strip() else [],
        "justification": txt.strip()[:300],
        "limitations": ["no clinical context", "not a validated medical model", "pediatric dataset"],
        "warning": WARNING_TEXT,
        "model_name": f"medgemma-4b-it-{cfg['name']}",
        "prompt_version": f"{cfg['name']}_v1", "latency_ms": latency_ms,
    }
    return apply_safety_guardrails(result)


processor, model = load_model()

mode = st.selectbox("Prompt", ["baseline", "improved"])

sample_dir = Path("data/real_cases")
samples = sorted([p for p in sample_dir.glob("*") if p.suffix.lower() in (".jpeg", ".jpg", ".png")]) if sample_dir.exists() else []

source = st.radio("Source de l'image", ["Exemple du depot", "Televerser"], horizontal=True)
image = None
if source == "Exemple du depot":
    if samples:
        choice = st.selectbox("Choisir une radio", [p.name for p in samples])
        image = Image.open(next(p for p in samples if p.name == choice)).convert("RGB")
    else:
        st.info("Aucune image dans data/real_cases/. Lance la cellule de telechargement, ou televerse une image.")
else:
    up = st.file_uploader("Deposer une radiographie thoracique frontale", type=["png", "jpg", "jpeg"])
    if up:
        image = Image.open(up).convert("RGB")

if image is not None:
    col1, col2 = st.columns([1, 1])
    with col1:
        st.image(image, caption="Image analysee", use_container_width=True)
    with col2:
        if st.button("Analyser avec MedGemma", type="primary"):
            with st.spinner("Analyse par MedGemma..."):
                pred = predire(image, PROMPTS[mode], processor, model)
            st.metric("Classe predite", pred["predicted_class"])
            st.metric("Confiance", pred["confidence"])
            st.metric("Latence (ms)", pred["latency_ms"])
            st.write("**Reponse brute du modele**")
            st.info(pred["justification"] or "(vide)")
            st.write("**Sortie JSON complete**")
            st.json(pred)


## 5. Lancer Streamlit + ouvrir le tunnel
La cellule affiche une adresse **https://....trycloudflare.com** : ouvre-la dans un nouvel onglet. **Ne stoppe pas cette cellule** pendant la demo (elle maintient le tunnel).

In [ ]:
import subprocess, time
subprocess.Popen(['streamlit', 'run', 'app_medgemma_colab.py',
                  '--server.port', '8501', '--server.headless', 'true'])
time.sleep(12)
print('Ouvre l URL trycloudflare.com qui apparait ci-dessous :')
!./cloudflared tunnel --url http://localhost:8501